# VietMedBridge — 00: Tải và kiểm tra ViBioMIR

Chạy trên Google Colab với runtime CPU. Notebook tải hai file Parquet của một
revision cố định, kiểm tra ID/schema và thống kê domain bằng DuckDB có disk spill.
Dữ liệu cùng checkpoint được lưu vào Google Drive; cần cấp quyền mount Drive ở cell đầu.

Dataset có hai cấu hình query và corpus. Split mang tên train là cách đóng gói
trên Hub; query hiện chỉ chứa id/query, chưa có nhãn reference để tính F2.
Giữ nguyên ID chính thức, kể cả khi ID không liên tục hoặc URL trùng nhau.


In [ ]:
import importlib.util
import json
import os
import subprocess
import sys
from pathlib import Path

if not (3, 11) <= sys.version_info[:2] < (3, 14):
    raise RuntimeError(
        "Dùng Python 3.11–3.13. Trong Runtime > Change runtime type, "
        "chọn phiên bản runtime có Python trong khoảng này."
    )
if importlib.util.find_spec("google.colab") is None:
    raise RuntimeError("Notebook này được thiết kế cho Google Colab.")

from google.colab import drive
# Chạy lại cell này sau MỖI runtime mới/restart: package chỉ được cài trong phiên Colab.
drive.mount("/content/drive")

# Giữ cùng DATA_ROOT trong cả bốn notebook.
DATA_ROOT = Path("/content/drive/MyDrive/VietMedBridge/data")
WORK_DIR = Path("/content/vmb_work")  # temp files/spill ở ổ local của Colab
DATA_ROOT.mkdir(parents=True, exist_ok=True)
WORK_DIR.mkdir(parents=True, exist_ok=True)
os.environ["HF_HOME"] = "/content/hf_cache"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

REPO_URL = "https://github.com/Platypus27-coder/VietMedBridge.git"
CODE_REVISION = None  # Có thể đặt full commit SHA; mặc định dùng code_lock đã lưu.
EXPECTED_PIPELINE_API = 2
CHECKOUT = Path("/content/VietMedBridge")
lock_path = DATA_ROOT / "code_lock.json"
lock = json.loads(lock_path.read_text()) if lock_path.exists() else {}
if lock and lock.get("repo_url") != REPO_URL:
    raise ValueError("DATA_ROOT đang khóa vào một repo khác.")
if lock and lock.get("pipeline_api") != EXPECTED_PIPELINE_API and not CODE_REVISION:
    raise RuntimeError(
        "DATA_ROOT đang dùng code cũ. Để nâng lên data-v2, đặt CODE_REVISION='main' "
        "ở cell này rồi chạy lại; dùng tên crawl/build run mới. Raw cũ vẫn được giữ."
    )
reference = CODE_REVISION or lock.get("git_commit") or "main"
if not (CHECKOUT / ".git").exists():
    subprocess.run(["git", "clone", "--no-checkout", REPO_URL, str(CHECKOUT)], check=True)
remote = subprocess.check_output(
    ["git", "-C", str(CHECKOUT), "remote", "get-url", "origin"], text=True
).strip()
if remote != REPO_URL:
    raise ValueError("Checkout hiện tại không thuộc repo VietMedBridge.")
subprocess.run(["git", "-C", str(CHECKOUT), "fetch", "--depth", "1", "origin", reference], check=True)
CODE_COMMIT = subprocess.check_output(
    ["git", "-C", str(CHECKOUT), "rev-parse", "FETCH_HEAD"], text=True
).strip()
if "vietmedbridge" in sys.modules and globals().get("_VMB_IMPORTED_COMMIT") != CODE_COMMIT:
    raise RuntimeError("Mã nguồn đổi trong runtime đã import package. Restart session rồi chạy lại notebook.")
subprocess.run(["git", "-C", str(CHECKOUT), "checkout", "--detach", "FETCH_HEAD"], check=True)
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-e", ".[notebook]"],
    cwd=CHECKOUT,
    check=True,
)
PACKAGE_SOURCE = CHECKOUT / "src"
if not (PACKAGE_SOURCE / "vietmedbridge" / "__init__.py").is_file():
    raise RuntimeError(
        f"Repo checkout thiếu package source: {PACKAGE_SOURCE}; commit={CODE_COMMIT}. "
        "Kiểm tra lại git fetch/checkout ở output của cell Bootstrap."
    )
sys.path.insert(0, str(PACKAGE_SOURCE))
importlib.invalidate_caches()
from vietmedbridge.artifacts import atomic_json, runtime_versions
from vietmedbridge import PIPELINE_API_VERSION
if PIPELINE_API_VERSION != EXPECTED_PIPELINE_API:
    raise RuntimeError("Notebook và package API không khớp; chọn commit data-v2 và restart session.")
_VMB_IMPORTED_COMMIT = CODE_COMMIT
if not lock or CODE_REVISION:
    atomic_json(lock_path, {"repo_url": REPO_URL, "git_commit": CODE_COMMIT, "pipeline_api": PIPELINE_API_VERSION})
PIPELINE_CONFIG = json.loads((CHECKOUT / "configs/data_pipeline.json").read_text())
atomic_json(DATA_ROOT / "runtime.json", {
    "git_commit": CODE_COMMIT, "python": sys.version, "packages": runtime_versions()
})
print("Code commit:", CODE_COMMIT)
import vietmedbridge
print("Package source:", Path(vietmedbridge.__file__).resolve())
print("Dữ liệu/checkpoint:", DATA_ROOT)


## 1. Tải snapshot gắn revision và kiểm tra SHA-256


In [ ]:
from vietmedbridge.dataset import DATASET_REVISION, snapshot_dataset, audit_dataset

# Revision đã kiểm tra của AIGuruTinix/ViBioMIR. Đổi revision sẽ tạo snapshot mới.
SNAPSHOT = snapshot_dataset(DATA_ROOT, revision=DATASET_REVISION)
print("Dataset revision:", SNAPSHOT["revision"])
for name, entry in SNAPSHOT["files"].items():
    print(name, entry.get("rows"), entry["bytes"], entry["sha256"])


## 2. Audit trực tiếp file Parquet

Không coi số dòng trong dataset card là số dòng đã xác minh. Báo cáo ghi số
dòng, số ID duy nhất, ID min/max, URL trùng, domain và độ dài query thực tế.
Sample chọn vài ID đầu của các domain lớn để thử crawler, không dùng làm tập đánh giá.


In [ ]:
REPORT = audit_dataset(DATA_ROOT, work_dir=WORK_DIR, sample_domains=30, per_domain=3)
print(json.dumps({key: REPORT[key] for key in ("query", "corpus", "sample")}, ensure_ascii=False, indent=2))
import pandas as pd
display(pd.DataFrame(REPORT["top_domains"]))


## Stage A: inventory, mẫu phân tầng và golden regression

Mẫu ~1.000 URL cân bằng nhóm domain, định dạng và language hint từ URL.
Đây là mẫu feasibility; tỷ lệ không trọng số chưa phải dự báo toàn corpus.
**Kiểm chứng nhanh:** output của cell Bootstrap phải hiện `Code commit` và
`Package source` dưới `/content/VietMedBridge/src/vietmedbridge`. Nếu setup
dừng ở lệnh pip/git, xử lý lỗi hiển thị tại đó trước khi chạy cell này.
Khi mở runtime mới hoặc vừa restart, chạy lại cell **Bootstrap** đầu notebook
(mount Drive, git clone/fetch và pip install) trước khi chạy cell này.
Golden suite dùng 11 fixture synthetic offline để kiểm tra trước pilot. Team cần
gán expected assertions cho 100–500 nguồn thật và replay ở notebook 03 trước Stage B1.
Đây là bước bootstrap golden thực tế từ nguồn đã crawl, chưa hoàn tất golden set của plan.


In [ ]:
import importlib.util
if importlib.util.find_spec("vietmedbridge") is None:
    raise RuntimeError(
        "Chưa setup package trong runtime này. Chạy cell Bootstrap đầu notebook "
        "để git clone/cập nhật VietMedBridge và cài package, rồi chạy lại cell này."
    )
from vietmedbridge.inventory import create_stage_a_sample
from vietmedbridge.golden import run_golden_suite
from vietmedbridge.chunks import ChunkConfig, load_bge_tokenizer
from vietmedbridge.artifacts import read_json, atomic_json

STAGE_A = create_stage_a_sample(DATA_ROOT, work_dir=WORK_DIR, **PIPELINE_CONFIG["stage_a"])
tokenizer_lock_path = DATA_ROOT / "tokenizer_lock.json"
tokenizer_lock = read_json(tokenizer_lock_path) if tokenizer_lock_path.exists() else {}
TOKENIZER, TOKENIZER_SPEC = load_bge_tokenizer(tokenizer_lock.get("revision"))
atomic_json(tokenizer_lock_path, TOKENIZER_SPEC)
GOLDEN = run_golden_suite(CHECKOUT / "tests/golden/cases.json", TOKENIZER, TOKENIZER_SPEC,
                          ChunkConfig(**PIPELINE_CONFIG["chunking"]))
atomic_json(DATA_ROOT / "reports/golden_latest.json", GOLDEN)
print("Stage A rows:", STAGE_A["rows"], "| golden:", GOLDEN["passed"])
display(pd.DataFrame(GOLDEN["cases"]))
if not GOLDEN["passed"]:
    raise RuntimeError("Golden regression fail; chưa được crawl/scale.")


## 3. Cách dùng load_dataset với đúng cấu hình

Query nhỏ được tải bình thường; corpus dùng streaming và chỉ xem ba dòng.
Corpus này là danh sách liên kết, chưa phải nội dung văn bản đã crawl.


In [ ]:
from datasets import load_dataset
from itertools import islice

queries = load_dataset(
    "AIGuruTinix/ViBioMIR", "query", split="train",
    revision=SNAPSHOT["revision"], cache_dir="/content/hf_cache/datasets",
)
corpus_preview = load_dataset(
    "AIGuruTinix/ViBioMIR", "corpus", split="train",
    revision=SNAPSHOT["revision"], streaming=True,
)
print("Queries:", len(queries), queries.features)
display(pd.DataFrame(islice(corpus_preview, 3)))


## Kết quả để dùng ở notebook 01

Google Drive/VietMedBridge/data/raw chứa snapshot nguồn có hash.
reports/dataset_audit.json chứa kết quả audit; reports/domains.parquet chứa
thống kê domain và reports/sample_links.parquet chứa mẫu thử crawler.

Lần đầu code_lock.json lưu commit GitHub đang chạy. Các notebook sau dùng lại
commit này. Khi chủ động nâng code, đặt CODE_REVISION và dùng tên crawl/build run mới.
Nguồn: [ViBioMIR](https://huggingface.co/datasets/AIGuruTinix/ViBioMIR).
